In [2]:
"""Cuestionario al agricultor - TerraQuipu (microcuenca Piuray-Qorimarca).

Bloques:
  esencial        (1, 2, 2b): si difieren de NASA y SENAMHI, se prioriza al agricultor.
  complementaria  (3 a 7):    se mezclan con los datos de NASA y SENAMHI.
  extra           (8 y 9):    la IA los usa al generar l1a matriz de calendarios.

Uso por consola:  python cuestionario_terraquipu.py
Uso desde una API o pruebas: ejecutar_cuestionario(responder=mi_funcion)
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Callable


# ----------------------------------------------------------------------
# Modelo
# ----------------------------------------------------------------------
@dataclass(frozen=True)
class Opcion:
    clave: str
    texto: str


@dataclass
class Pregunta:
    id: str
    bloque: str  # "esencial" | "complementaria" | "extra"
    texto: str
    opciones: list[Opcion]
    # Recibe las respuestas dadas hasta ese momento; si devuelve False se omite.
    condicion: Callable[[dict], bool] = lambda respuestas: True
    # clave de opcion -> pregunta que se abre al elegirla
    abre: dict[str, "Pregunta"] = field(default_factory=dict)


# ----------------------------------------------------------------------
# Componente reutilizable: desplegable de cultivo (preguntas 1, 2b y 7)
# ----------------------------------------------------------------------
CULTIVOS = [
    Opcion("papa", "Papa"),
    Opcion("olluco", "Olluco"),
    Opcion("quinua", "Quinua"),
    Opcion("oca", "Oca"),
    Opcion("otro", "Otro cultivo"),
]

TIPOS_DE_CULTIVO = [
    Opcion("tuberculo_raiz", "Tubérculo / Raíz"),
    Opcion("leguminosa", "Leguminosa (Vaina)"),
    Opcion("cereal_grano", "Cereal / Grano"),
    Opcion("hortaliza", "Hortaliza"),
]

# Tipo de los cultivos de la lista. La quinua es un seudocereal: aquí se agrupa
# como cereal/grano (confirmar con el equipo).
TIPO_POR_CULTIVO = {
    "papa": "tuberculo_raiz",
    "olluco": "tuberculo_raiz",
    "oca": "tuberculo_raiz",
    "quinua": "cereal_grano",
}


def pregunta_cultivo(id_, bloque, texto, extra=(), condicion=lambda r: True):
    """Desplegable de cultivo; si elige 'Otro cultivo' se abre '¿Qué tipo es?'."""
    tipo = Pregunta(f"{id_}_tipo", bloque, "¿Qué tipo de cultivo es?", TIPOS_DE_CULTIVO)
    return Pregunta(id_, bloque, texto, CULTIVOS + list(extra), condicion, abre={"otro": tipo})


DESCANSO = Opcion("descanso", "Descansó")

# ----------------------------------------------------------------------
# Preguntas (en el orden en que se muestran)
# ----------------------------------------------------------------------
P1 = Pregunta(
    "1", "esencial", "¿Qué hay en su chacra hoy?",
    [
        Opcion("cultivo_creciendo", "Cultivo creciendo"),
        Opcion("rastrojo", "Sin cultivo, pero con restos secos de la cosecha (paja, tallos)"),
        Opcion("pasto", "Pasto o hierba"),
        Opcion("tierra_desnuda", "Solo tierra, sin nada encima"),
    ],
    abre={"cultivo_creciendo": pregunta_cultivo("1a", "esencial", "¿Cuál?")},
)

# Solo corre si en la 1 eligió una de las tres últimas opciones.
P2 = Pregunta(
    "2", "esencial", "¿Cuándo piensa sembrar?",
    [
        Opcion("esta_semana", "Empiezo esta semana"),
        Opcion("dos_a_cuatro_semanas", "En unas 2 a 4 semanas"),
        Opcion("mas_de_un_mes", "Dentro de más de un mes"),
        Opcion("descansar", "Esta vez la voy a dejar descansar"),
    ],
    condicion=lambda r: r["1"] != "cultivo_creciendo",
)

# Se omite si no se hizo la 2 o si en la 2 respondió que la dejará descansar.
P2B = pregunta_cultivo(
    "2b", "esencial", "¿Qué piensa sembrar?",
    condicion=lambda r: r.get("2") not in (None, "descansar"),
)

P3 = Pregunta(
    "3", "complementaria", "¿Cómo recibe agua su chacra?",
    [
        Opcion("solo_lluvia", "Solo cuando llueve"),
        Opcion("lluvia_y_auxilio", "Cuando llueve, y a veces puedo regar si hace falta"),
        Opcion("riego_permanente", "Tengo riego todo el año (canal, reservorio, aspersor)"),
    ],
)

P4 = Pregunta(
    "4", "complementaria",
    "¿Su parcela cuenta con defensas contra las lluvias, "
    "como zanjas, andenes o barreras de plantas?",
    [
        Opcion("buen_estado", "Sí, y están en buen estado"),
        Opcion("mal_estado", "Sí, pero están en mal estado"),
        Opcion("ninguna", "No tengo"),
    ],
)

P5 = Pregunta(
    "5", "complementaria", "Cuando cava con la lampa, ¿cómo es la tierra?",
    [
        Opcion("profundo", "Puedo cavar hondo sin problema"),
        Opcion("algo_piedras", "Hay algunas piedras, pero igual se puede cavar"),
        Opcion("muy_pedregoso", "Hay muchas piedras o se topa roca pronto"),
    ],
)

P6 = Pregunta(
    "6", "complementaria", "¿Sabe qué tipo de tierra tiene su chacra?",
    [Opcion("conoce", "Sí, la conozco"), Opcion("no_seguro", "No estoy seguro")],
    abre={
        "conoce": Pregunta(
            "6a", "complementaria", "¿Cómo es?",
            [
                Opcion("suelta", "Suelta, se deshace fácil"),
                Opcion("suave", "Suave y buena para trabajar"),
                Opcion("pesada", "Pesada y pegajosa cuando está mojada"),
            ],
        ),
        "no_seguro": Pregunta(
            "6b", "complementaria",
            "Coja un puñado de tierra húmeda y apriételo. ¿Qué pasa?",
            [
                Opcion("se_desmorona", "Se desmorona y raspa"),
                Opcion("bolita", "Se hace una bolita suave que se mantiene"),
                Opcion("barro", "Se pega como barro y se queda marcada la mano"),
            ],
        ),
    },
)

P7A = pregunta_cultivo(
    "7a", "complementaria",
    "¿Qué sembró en esta chacra la vez anterior?",
    extra=[DESCANSO],
)
P7B = pregunta_cultivo(
    "7b", "complementaria",
    "¿Qué sembró en esta chacra la penultima vez? · Y antes de esa",
    extra=[DESCANSO],
)

P8 = Pregunta(
    "8", "extra", "¿Cuántas personas pueden trabajar en la chacra para prepararla?",
    [
        Opcion("solo_yo", "Solo yo"),
        Opcion("dos_o_tres", "2 o 3 personas"),
        Opcion("cuatro_o_mas", "4 o más personas"),
    ],
)

P9 = Pregunta(
    "9", "extra", "¿Y cuánto tiempo puede dedicar cada semana?",
    [
        Opcion("poco", "Poco, algunas horas"),
        Opcion("algo", "Algo, uno o dos días"),
        Opcion("bastante", "Bastante, tres días o más"),
    ],
)

PREGUNTAS = [P1, P2, P2B, P3, P4, P5, P6, P7A, P7B, P8, P9]


# ----------------------------------------------------------------------
# Ejecución del flujo
# ----------------------------------------------------------------------
def responder_por_consola(pregunta: Pregunta) -> str:
    print(f"\n[{pregunta.id}] {pregunta.texto}")
    for i, opcion in enumerate(pregunta.opciones, start=1):
        print(f"  {i}. {opcion.texto}")
    while True:
        entrada = input("Elija una opción: ").strip()
        if entrada.isdigit() and 1 <= int(entrada) <= len(pregunta.opciones):
            return pregunta.opciones[int(entrada) - 1].clave
        print("Opción no válida. Escriba el número de una de las opciones.")


def _ejecutar(pregunta: Pregunta, respuestas: dict, responder: Callable) -> None:
    if not pregunta.condicion(respuestas):
        return
    clave = responder(pregunta)
    if clave not in {o.clave for o in pregunta.opciones}:
        raise ValueError(f"Respuesta '{clave}' no válida para la pregunta {pregunta.id}")
    respuestas[pregunta.id] = clave
    if clave in pregunta.abre:
        _ejecutar(pregunta.abre[clave], respuestas, responder)


def ejecutar_cuestionario(responder: Callable = responder_por_consola,
                          preguntas: list[Pregunta] = PREGUNTAS) -> dict:
    """Recorre el cuestionario y devuelve {id_pregunta: clave_de_respuesta}."""
    respuestas: dict = {}
    for pregunta in preguntas:
        _ejecutar(pregunta, respuestas, responder)
    return respuestas


# ----------------------------------------------------------------------
# De respuestas a parámetros
# ----------------------------------------------------------------------
COBERTURA = {
    "cultivo_creciendo": "cultivo",
    "rastrojo": "rastrojo",
    "pasto": "pasto",
    "tierra_desnuda": "suelo_desnudo",
}
AGUA = {
    "solo_lluvia": "secano",
    "lluvia_y_auxilio": "riego_auxilio",
    "riego_permanente": "riego_permanente",
}
TEXTURA = {
    "suelta": "arenosa", "se_desmorona": "arenosa",
    "suave": "franca", "bolita": "franca",
    "pesada": "arcillosa", "barro": "arcillosa",
}


def _cultivo(respuestas: dict, id_: str):
    clave = respuestas.get(id_)
    if clave is None:
        return None
    if clave == "descanso":
        return {"cultivo": "descanso", "tipo": None}
    if clave == "otro":
        return {"cultivo": "otro", "tipo": respuestas.get(f"{id_}_tipo")}
    return {"cultivo": clave, "tipo": TIPO_POR_CULTIVO[clave]}


def construir_parametros(r: dict) -> dict:
    """Agrupa las respuestas por bloque y las convierte en parámetros del algoritmo."""
    esenciales = {"cobertura": COBERTURA[r["1"]]}
    if r["1"] == "cultivo_creciendo":
        esenciales["cultivo_actual"] = _cultivo(r, "1a")
    else:
        esenciales["cultivo_actual"] = None
        esenciales["siembra"] = r["2"]
        esenciales["cultivo_proximo"] = (
            {"cultivo": "descanso", "tipo": None} if r["2"] == "descansar" else _cultivo(r, "2b")
        )

    textura_clave = r.get("6a") or r.get("6b")
    complementarias = {
        "agua": AGUA[r["3"]],
        "obras": r["4"],
        "profundidad": r["5"],
        "bloquea_obras_profundas": r["5"] == "muy_pedregoso",
        "textura": {
            "clase": TEXTURA[textura_clave],
            "origen": "declarado" if r["6"] == "conoce" else "prueba_del_punado",
        },
        "historial": {"vez_anterior": _cultivo(r, "7a"), "antes_de_esa": _cultivo(r, "7b")},
    }

    extra = {"personas": r["8"], "tiempo_semanal": r["9"]}
    return {"esenciales": esenciales, "complementarias": complementarias, "factores_extra": extra}


def reconciliar_esenciales(del_agricultor: dict, externos: dict) -> tuple[dict, dict]:
    """Si un parámetro esencial difiere del de NASA/SENAMHI, gana el agricultor.

    Devuelve (parametros_finales, discrepancias). Las discrepancias sirven como
    dato de validación. Las claves que no se preguntaron no aparecen en
    `del_agricultor`, así que conservan el valor externo.
    """
    finales = dict(externos)
    discrepancias = {}
    for clave, valor in del_agricultor.items():
        if clave in externos and externos[clave] != valor:
            discrepancias[clave] = {"agricultor": valor, "externo": externos[clave]}
        finales[clave] = valor
    return finales, discrepancias


if __name__ == "__main__":
    respuestas = ejecutar_cuestionario()
    parametros = construir_parametros(respuestas)
    print(json.dumps({"respuestas": respuestas, "parametros": parametros},
                     ensure_ascii=False, indent=2))


[1] ¿Qué hay en su chacra hoy?
  1. Cultivo creciendo
  2. Sin cultivo, pero con restos secos de la cosecha (paja, tallos)
  3. Pasto o hierba
  4. Solo tierra, sin nada encima
Elija una opción: 1

[1a] ¿Cuál?
  1. Papa
  2. Olluco
  3. Quinua
  4. Oca
  5. Otro cultivo
Elija una opción: 5

[1a_tipo] ¿Qué tipo de cultivo es?
  1. Tubérculo / Raíz
  2. Leguminosa (Vaina)
  3. Cereal / Grano
  4. Hortaliza
Elija una opción: 3

[3] ¿Cómo recibe agua su chacra?
  1. Solo cuando llueve
  2. Cuando llueve, y a veces puedo regar si hace falta
  3. Tengo riego todo el año (canal, reservorio, aspersor)
Elija una opción: 1

[4] ¿Su parcela cuenta con defensas contra las lluvias, como zanjas, andenes o barreras de plantas?
  1. Sí, y están en buen estado
  2. Sí, pero están en mal estado
  3. No tengo
Elija una opción: 2

[5] Cuando cava con la lampa, ¿cómo es la tierra?
  1. Puedo cavar hondo sin problema
  2. Hay algunas piedras, pero igual se puede cavar
  3. Hay muchas piedras o se topa roca